In [ ]:
# Setup: credentials from .env (never hardcode)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from notebooks.load_env import load_env, get_project_root
load_env()
PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "archive"

In [ ]:
# !uv sync

# 01 — Prepare Postgres Data (Vector Store)

News chunks + embeddings for PGVector. **Prerequisites:** `.env` configured, `docker compose up -d`.

---

## 1. Load and Clean Data

In [3]:
content_path = DATA_DIR / "data.csv"
rating_path = DATA_DIR / "rating.csv"

In [4]:
import pandas as pd

# 1. Load the data
df_content = pd.read_csv(content_path)
df_rating = pd.read_csv(rating_path)

# 2. Drop rows missing 'full_content'
df_clean = df_content.dropna(subset=['full_content']).copy()

# 3. Merge with ratings to get sentiment labels
# Note: how='inner' ensures we only keep articles that have both text AND a sentiment label
final_df = df_clean.merge(
    df_rating[['article_id', 'title_sentiment']], 
    on='article_id', 
    how='inner'
)

# 4. Final Cleanup: Select only necessary columns to keep Postgres lean
columns_to_keep = [
    'article_id', 'source_name', 'author', 'title', 
    'published_at', 'full_content', 'category', 'title_sentiment'
]
final_df = final_df[columns_to_keep]

print(f"Original size: {len(df_content)}")
print(f"Cleaned size: {len(final_df)}")

Original size: 105375
Cleaned size: 63836


#### Sampling 5k pilot - by published time and by sentiment distribution

In [5]:
# 1. Ensure published_at is a datetime object (coerce invalid → NaT, then drop)
final_df['published_at'] = pd.to_datetime(final_df['published_at'], errors='coerce')
final_df = final_df.dropna(subset=['published_at'])

# 2. Stratified Sampling by Sentiment
# This takes an equal number of rows for each sentiment type
pilot_df = final_df.groupby('title_sentiment', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 1666), random_state=42)
)

# 3. Sort by date to maintain chronological integrity for the agent
pilot_df = pilot_df.sort_values('published_at').reset_index(drop=True)

print(f"Pilot size: {len(pilot_df)}")
print(pilot_df['title_sentiment'].value_counts())

Pilot size: 4998
title_sentiment
Positive    1666
Negative    1666
Neutral     1666
Name: count, dtype: int64


/var/folders/zj/yysjfhvx241gg2nhx1b69xq40000gn/T/ipykernel_1296/3108844760.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pilot_df = final_df.groupby('title_sentiment', group_keys=False).apply(


In [6]:
pilot_df

,article_id,source_name,author,title,published_at,full_content,category,title_sentiment
0,20287,Android Central,namerah.saud-fatmi@futurenet.com (Namerah Saud...,iOttie iON Wireless Duo charger stand review: ...,2023-10-01 20:00:46,There are loads of weird and wonderful wireles...,Google,Positive
1,24887,Al Jazeera English,Al Jazeera,"At least 10 migrants killed, 25 injured in Mex...",2023-10-01 20:22:17,Thousands of migrants from different countries...,Travel,Negative
2,31352,BBC News,https://www.facebook.com/bbcnews,Why government shutdowns seem to only happen i...,2023-10-02 00:04:05,The US government has shutdown ten times over ...,Food,Negative
3,30910,The Indian Express,Express News Service,Today in Politics: Modi focus on infra stimulu...,2023-10-02 02:04:23,Money never sleeps and so don’t politicians as...,Politics,Neutral
4,26202,Android Central,michael.hicks@futurenet.com (Michael L Hicks),Quest 3 will succeed at fitness where the Ques...,2023-10-02 02:25:42,"At Meta Connect 2023, most people who tried th...",Fitness,Positive
...,...,...,...,...,...,...,...,...
4993,130737,Digital Trends,Jennifer Allen,These new Chromebook Plus laptops are already ...,2023-11-03 15:09:25,With the launch of new Chromebook Plus laptops...,Design,Positive
4994,132634,Forbes,"Brendan Ahern, Senior Contributor, \n Brendan ...","Growth Stocks Rebound, Humanoid Robots Promote...",2023-11-03 15:09:31,CLN Asian equities ended a strong week highe...,Asia,Positive
4995,133130,Forbes,"Hugh McIntyre, Senior Contributor, \n Hugh McI...",Taylor Swift Breaks The Record For The Largest...,2023-11-03 15:15:22,"Inglewood, CA - August 07: Taylor Swift perfor...",America,Positive
4996,132272,Digital Trends,Jennifer Allen,Best TV deals: Get a 65-inch 4K TV for under $...,2023-11-03 15:17:22,Is it time to upgrade your home theater system...,world,Positive


## Create Embeddings for the pilot data: full_content + title

In [7]:
# Word count stats for full_content (for chunking decisions) — BEFORE truncation
word_counts = pilot_df['full_content'].str.split().str.len()

print("=== full_content: Word count descriptive stats (pre-truncation) ===\n")
print(f"{'Count':<14} {len(word_counts):,}")
print(f"{'Min':<14} {word_counts.min():,.0f}")
print(f"{'Max':<14} {word_counts.max():,.0f}")
print(f"{'Mean':<14} {word_counts.mean():,.1f}")
print(f"{'Median':<14} {word_counts.median():,.0f}")
print(f"{'Std':<14} {word_counts.std():,.1f}")
print("\nPercentiles:")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p:<3}  {word_counts.quantile(p/100):,.0f}")

=== full_content: Word count descriptive stats (pre-truncation) ===

Count          4,998
Min            11
Max            36,782
Mean           1,037.7
Median         542
Std            2,548.8

Percentiles:
  P10   246
  P25   355
  P50   542
  P75   878
  P90   1,408
  P95   2,534
  P99   12,362


### Truncate full_content to max 3k words <P95

In [8]:
# Truncate full_content to max 5k words
MAX_WORDS = 3000

def truncate_to_words(text, max_words=MAX_WORDS):
    if pd.isna(text):
        return text
    words = str(text).split()
    return " ".join(words[:max_words]) if len(words) > max_words else text

pilot_df['full_content'] = pilot_df['full_content'].apply(truncate_to_words)

truncated = pilot_df['full_content'].str.split().str.len()
print(f"Truncated to max {MAX_WORDS:,} words:")
print(f"  Max now: {truncated.max():,.0f} | Articles at limit: {(truncated >= MAX_WORDS).sum():,}")

Truncated to max 3,000 words:
  Max now: 3,000 | Articles at limit: 213


In [9]:
# Minimally Clean text
import re

def clean_for_embedding(text):
    # 1. Remove URLs and Emails
    text = re.sub(r'\S*@\S*\s?', '', text) 
    text = re.sub(r'http\S+', '', text)
    
    # 2. Remove common news "Noise" phrases (Customize this list)
    noise_phrases = ["Click here to read more", "Follow us on social media", "Advertisement"]
    for phrase in noise_phrases:
        text = text.replace(phrase, "")
        
    # 3. Standardize whitespace
    text = " ".join(text.split())
    return text

# Apply to your Pilot DF
pilot_df['full_content'] = pilot_df['full_content'].apply(clean_for_embedding)

In [10]:
# CHUCKING HERE 

# Set chunking params for pilot
CHUNK_SIZE = 1000   # chars (~250 tokens; fits <256 context)
CHUNK_OVERLAP = 100 # chars (~25 tokens); small, minimizes redundancy

def fast_safe_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):

    if pd.isna(text) or not text.strip():
        return []

    chunks = []
    text = text.strip()
    n = len(text)
    start = 0

    while start < n:

        end = start + chunk_size

        if end >= n:
            chunks.append(text[start:].strip())
            break

        chunk_end = text.rfind(' ', start, end)

        if chunk_end == -1 or chunk_end <= start:
            actual_end = end
        else:
            actual_end = chunk_end

        chunks.append(text[start:actual_end].strip())

        # Move start with overlap
        start = max(actual_end - overlap, 0)

        # Snap start to word boundary
        next_space = text.find(' ', start)
        if next_space != -1 and next_space < actual_end:
            start = next_space + 1

    return [c for c in chunks if len(c) > 10]

# Apply chunking to the pilot data
pilot_df['content_chunks'] = pilot_df['full_content'].apply(fast_safe_chunks)

#### Create embeddings

In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from tqdm.auto import tqdm
import time

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

# Explode chunks
chunks_df = (
    pilot_df[['content_chunks']]
    .explode('content_chunks', ignore_index=True)
    .rename(columns={'content_chunks': 'chunk_text'})
)

texts = chunks_df["chunk_text"].tolist()

print(f"Embedding {len(texts)} chunks on {device}...")

batch_size = 128
embeddings = []

start_time = time.time()

for i in tqdm(range(0, len(texts), batch_size), desc="Embedding progress"):
    
    batch = texts[i:i + batch_size]

    batch_emb = model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    embeddings.append(batch_emb)

embeddings = np.vstack(embeddings)

elapsed = time.time() - start_time
print(f"\nEmbedding completed in {elapsed:.2f} seconds")

chunks_df["embedding"] = list(embeddings)

/Users/shrinivaskallol/src/agentic-media-intelligence/agentic-media-intelligence/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding 27937 chunks on cpu...


Embedding progress:   0%|          | 1/219 [00:16<1:00:06, 16.55s/it]


KeyboardInterrupt: 

In [ ]:
# Upsert to Postgres — credentials from .env
import os
import pandas as pd
import psycopg2

df = pd.read_parquet(PROJECT_ROOT / "notebooks" / "pilot_chunks_with_embeddings.parquet", engine="fastparquet")

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5433")),
    database=os.getenv("POSTGRES_DB", "market_intel"),
    user=os.getenv("POSTGRES_USER", "admin"),
    password=os.getenv("POSTGRES_PASSWORD"),
)

cur = conn.cursor()
print("Connected to Postgres")

Connected to Postgres


In [13]:
pilot_chunks = (
    pilot_df
    .explode("content_chunks", ignore_index=True)
    .rename(columns={"content_chunks": "chunk_text"})
)


In [14]:
final_df = pilot_chunks.merge(
    chunks_df[["chunk_text", "embedding"]],
    on="chunk_text",
    how="left"
)

KeyError: "['embedding'] not in index"

In [25]:
final_df.to_csv("pilot_chunks_with_embeddings.csv", index=False)

In [ ]:
final_df["embedding"] = final_df["embedding"].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else x
)

final_df.to_parquet(
    "pilot_chunks_with_embeddings.parquet",
    engine="fastparquet",
    compression="snappy",
    index=False
)

print("Saved successfully.")

In [15]:
import pandas as pd
df = pd.read_parquet(
    "pilot_chunks_with_embeddings.parquet",
    engine="fastparquet"
)

df

,article_id,source_name,author,title,published_at,full_content,category,title_sentiment,chunk_text,embedding
0,20287,Android Central,namerah.saud-fatmi@futurenet.com (Namerah Saud...,iOttie iON Wireless Duo charger stand review: ...,2023-10-01 20:00:46,There are loads of weird and wonderful wireles...,Google,Positive,There are loads of weird and wonderful wireles...,"[-0.11936783045530319, 0.05921657010912895, 0...."
1,20287,Android Central,namerah.saud-fatmi@futurenet.com (Namerah Saud...,iOttie iON Wireless Duo charger stand review: ...,2023-10-01 20:00:46,There are loads of weird and wonderful wireles...,Google,Positive,charger stand costs around $50. That's a lot c...,"[-0.15836475789546967, 0.06643933802843094, 0...."
2,20287,Android Central,namerah.saud-fatmi@futurenet.com (Namerah Saud...,iOttie iON Wireless Duo charger stand review: ...,2023-10-01 20:00:46,There are loads of weird and wonderful wireles...,Google,Positive,recharge your earbuds wirelessly at 5W. You co...,"[-0.03099595569074154, 0.009008830413222313, 0..."
3,20287,Android Central,namerah.saud-fatmi@futurenet.com (Namerah Saud...,iOttie iON Wireless Duo charger stand review: ...,2023-10-01 20:00:46,There are loads of weird and wonderful wireles...,Google,Positive,"household, I didn't put this to the test, but ...","[-0.14629895985126495, 0.018404820933938026, 0..."
4,20287,Android Central,namerah.saud-fatmi@futurenet.com (Namerah Saud...,iOttie iON Wireless Duo charger stand review: ...,2023-10-01 20:00:46,There are loads of weird and wonderful wireles...,Google,Positive,prospect is undeniable. You can charge two dev...,"[-0.14222440123558044, 0.011647882871329784, 0..."
...,...,...,...,...,...,...,...,...,...,...
28448,134902,The Times of India,ANI,Climate change threatens to reverse health gai...,2023-11-03 15:21:45,ANI photo GENEVA: As the world warms at a fast...,Africa,Negative,health and climate sectors are needed. In the ...,"[-0.033733248710632324, -0.00478852353990078, ..."
28449,134902,The Times of India,ANI,Climate change threatens to reverse health gai...,2023-11-03 15:21:45,ANI photo GENEVA: As the world warms at a fast...,Africa,Negative,a strong double-digit year-on-year growth in t...,"[0.022529061883687973, 0.013201962225139141, -..."
28450,134902,The Times of India,ANI,Climate change threatens to reverse health gai...,2023-11-03 15:21:45,ANI photo GENEVA: As the world warms at a fast...,Africa,Negative,a strong double-digit year-on-year growth in t...,"[0.022529061883687973, 0.013201962225139141, -..."
28451,134902,The Times of India,ANI,Climate change threatens to reverse health gai...,2023-11-03 15:21:45,ANI photo GENEVA: As the world warms at a fast...,Africa,Negative,of India (Sebi) for the alleged diversion of f...,"[-0.01735387183725834, -0.06587207317352295, -..."


### Push data to pgvector store 

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd
import numpy as np

# 1. Load dataset
df = pd.read_parquet("pilot_chunks_with_embeddings.parquet", engine="fastparquet")

# 2. Database Connection
conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"), port=int(os.getenv("POSTGRES_PORT", "5433")), database=os.getenv("POSTGRES_DB", "market_intel"), user=os.getenv("POSTGRES_USER", "admin"), password=os.getenv("POSTGRES_PASSWORD")
)
cur = conn.cursor()

# 3. Schema Setup - FORCE REFRESH
print("Cleaning old schema and preparing new table...")
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
# We drop the table because IF NOT EXISTS won't update columns if the table already exists
cur.execute("DROP TABLE IF EXISTS article_chunks CASCADE;") 

cur.execute("""
CREATE TABLE article_chunks (
    article_id TEXT NOT NULL,
    source_name TEXT,
    author TEXT,
    title TEXT,
    published_at TIMESTAMPTZ,
    category TEXT,
    title_sentiment TEXT,
    content TEXT NOT NULL,
    embedding vector(384),
    fts_tokens tsvector,
    chunk_hash TEXT GENERATED ALWAYS AS (md5(article_id || '::' || content)) STORED,
    PRIMARY KEY (chunk_hash)
)
""")
conn.commit()

# 4. Pre-processing
def to_list(v):
    if isinstance(v, np.ndarray): return v.tolist()
    return v

df["embedding"] = df["embedding"].apply(to_list)
df = df.drop_duplicates(subset=["article_id", "chunk_text"], keep="first")

# 5. Data Mapping
records = [
    (
        str(r.article_id),
        r.source_name,
        r.author,
        r.title,
        r.published_at if pd.notnull(r.published_at) else None,
        r.category,
        r.title_sentiment,
        r.chunk_text, 
        r.embedding
    )
    for r in df.itertuples(index=False)
]

# 6. Upsert Logic
sql = """
INSERT INTO article_chunks (
    article_id, source_name, author, title, published_at, 
    category, title_sentiment, content, embedding
)
VALUES %s
ON CONFLICT (chunk_hash) DO UPDATE SET
    source_name = EXCLUDED.source_name,
    author = EXCLUDED.author,
    title = EXCLUDED.title,
    published_at = EXCLUDED.published_at,
    category = EXCLUDED.category,
    title_sentiment = EXCLUDED.title_sentiment,
    embedding = EXCLUDED.embedding
"""

print(f"Ingesting {len(records)} records...")
execute_values(cur, sql, records, page_size=1000)
conn.commit()

# 7. Post-Ingest: Generate FTS Tokens
print("Updating Full-Text Search tokens...")
cur.execute("""
    UPDATE article_chunks 
    SET fts_tokens = to_tsvector('english', coalesce(title, '') || ' ' || coalesce(content, ''));
""")

# 8. Create Indices for Performance
print("Creating HNSW index...")
cur.execute("CREATE INDEX ON article_chunks USING hnsw (embedding vector_cosine_ops);")
cur.execute("CREATE INDEX ON article_chunks USING gin(fts_tokens);")

conn.commit()
print("Done! Database is fully operational.")

cur.close()
conn.close()

Cleaning old schema and preparing new table...
Ingesting 27842 records...
Updating Full-Text Search tokens...
Creating HNSW index...
Done! Database is fully operational.


### Create pgvector IVFFlat index (run after upsert)

**Why this is crucial:** Vector search computes similarity `distance(query_embedding, stored_embedding)`.

- **Without index:** 27k vectors → every query checks all 27k rows
- **With IVFFlat:** vectors grouped into clusters → query searches only nearest clusters (e.g. ~278 per cluster with `lists=100` → ~300 rows vs 27k)

In [ ]:
# Create pgvector IVFFlat index for fast similarity search
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5433")),
    database=os.getenv("POSTGRES_DB", "market_intel"),
    user=os.getenv("POSTGRES_USER", "admin"),
    password=os.getenv("POSTGRES_PASSWORD"),
)
cur = conn.cursor()

cur.execute("""
CREATE INDEX IF NOT EXISTS article_chunks_embedding_idx
ON article_chunks
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);
""")
conn.commit()
print("✓ IVFFlat index created")

cur.execute("ANALYZE article_chunks")
conn.commit()
print("✓ ANALYZE complete")

cur.close()
conn.close()

✓ IVFFlat index created
✓ ANALYZE complete


### Hybrid Search (BM25 + Vector) and HNSW Index

**1. BM25 (Full-Text Search):** Dense vectors are great for semantics but weak on exact names (e.g. "iOttie", "RationalStat"). BM25 (sparse, keyword) fixes this.

**2. HNSW Index:** At 5k+ rows, HNSW is the 2026 standard—handles incremental updates (new news daily) without degrading recall. IVFFlat needs periodic rebuilds; HNSW does not.

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5433")),
    database=os.getenv("POSTGRES_DB", "market_intel"),
    user=os.getenv("POSTGRES_USER", "admin"),
    password=os.getenv("POSTGRES_PASSWORD"),
)
cur = conn.cursor()

# 1. Add TSVector column for BM25 (keyword) search
cur.execute("""
    ALTER TABLE article_chunks
    ADD COLUMN IF NOT EXISTS fts_tokens tsvector
    GENERATED ALWAYS AS (to_tsvector('english', content)) STORED;
""")
conn.commit()
print("✓ fts_tokens column added")

# 2. GIN index for FTS speed
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_fts ON article_chunks USING GIN (fts_tokens);
""")
conn.commit()
print("✓ GIN index on fts_tokens created")

# 3. HNSW index (graph-based, better for incremental updates)
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_hnsw_embedding ON article_chunks
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
""")
conn.commit()
print("✓ HNSW index created")

cur.execute("ANALYZE article_chunks")
conn.commit()
print("✓ ANALYZE complete")

cur.close()
conn.close()

✓ fts_tokens column added
✓ GIN index on fts_tokens created
✓ HNSW index created
✓ ANALYZE complete


### RRF Hybrid Smoke Test (BM25 + Vector)

**Reciprocal Rank Fusion (RRF)** merges results from multiple rankers (keyword + semantic) without score calibration. Each list contributes `1/(k + rank)` per doc; we sum across lists. The constant `k=60` dampens top-heavy effects. **Scores are not 0–1 or 1–100** — typical RRF scores are small (e.g. 0.01–0.05). Higher = better, but compare *relative* rank, not absolute.

| RRF Score | Interpretation | Action |
|-----------|-----------------|--------|
| > 0.030 | Golden Match | Hit #1 in both Keyword and Vector. Extremely relevant. |
| 0.015–0.025 | Strong Match | Likely the top hit in one search engine. Very reliable. |
| 0.010–0.015 | Relevant | Top 10–20 hit. Good for context, but not a "direct" answer. |
| < 0.010 | Noise | Likely a low-ranking filler result. Ignore for final LLM synthesis. |

- **Keyword-heavy:** "iOttie" — BM25 excels at exact brand names
- **Semantic-heavy:** "market trends in EV charging" — vectors capture concepts

In [23]:
import psycopg2
from sentence_transformers import SentenceTransformer

# 1. Setup
model = SentenceTransformer('all-MiniLM-L6-v2')
conn = psycopg2.connect(host=os.getenv("POSTGRES_HOST", "localhost"), port=int(os.getenv("POSTGRES_PORT", "5433")), database=os.getenv("POSTGRES_DB", "market_intel"), user=os.getenv("POSTGRES_USER", "admin"), password=os.getenv("POSTGRES_PASSWORD"))
cur = conn.cursor()

def smoke_test_hybrid(query_text):
    print(f"\n--- Testing Query: '{query_text}' ---")
    
    # Generate Embedding
    query_vector = model.encode(query_text).tolist()

    # RRF Hybrid Query
    # k=60 is the standard constant that balances top-ranked results
    hybrid_query = """
    WITH semantic_search AS (
        SELECT chunk_hash, ROW_NUMBER() OVER (ORDER BY embedding <=> %s::vector) as rank
        FROM article_chunks
        ORDER BY embedding <=> %s::vector
        LIMIT 20
    ),
    keyword_search AS (
        SELECT chunk_hash, ROW_NUMBER() OVER (ORDER BY ts_rank_cd(fts_tokens, plainto_tsquery('english', %s)) DESC) as rank
        FROM article_chunks
        WHERE fts_tokens @@ plainto_tsquery('english', %s)
        ORDER BY rank DESC
        LIMIT 20
    )
    SELECT 
        a.title,
        a.category,
        a.content,
        COALESCE(1.0 / (60 + s.rank), 0.0) + COALESCE(1.0 / (60 + k.rank), 0.0) AS rrf_score
    FROM semantic_search s
    FULL OUTER JOIN keyword_search k ON s.chunk_hash = k.chunk_hash
    JOIN article_chunks a ON a.chunk_hash = COALESCE(s.chunk_hash, k.chunk_hash)
    ORDER BY rrf_score DESC
    LIMIT 3;
    """

    cur.execute(hybrid_query, (query_vector, query_vector, query_text, query_text))
    results = cur.fetchall()

    if not results:
        print("No results found. Check if FTS tokens were generated.")
        return

    for i, (title, cat, content, score) in enumerate(results):
        print(f"{i+1}. [Score: {score:.4f}] [{cat}] {title}")
        print(f"   Snippet: {content[:150]}...\n")

# Run two distinct tests
smoke_test_hybrid("iOttie wireless charger price and safety")  # Keyword-heavy
smoke_test_hybrid("market trends in electronic vehicle charging")  # Semantic-heavy

cur.close()
conn.close()

/Users/shrinivaskallol/src/agentic-media-intelligence/agentic-media-intelligence/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



--- Testing Query: 'iOttie wireless charger price and safety' ---
1. [Score: 0.0164] [Google] iOttie iON Wireless Duo charger stand review: The best Made for Google charging accessory
   Snippet: charger stand costs around $50. That's a lot cheaper than Google's offering, and it doesn't stop there. Since this is a dual wireless charger, you get...

2. [Score: 0.0161] [Amazon] The best October Prime Day deals under $50
   Snippet: when you clip the on-page coupon. Kasa’s plugs allow you to add modern smarts to more traditional gadgets, thus giving you the ability to set schedule...

3. [Score: 0.0159] [Amazon] Prime Day deals under $100 — this is what I'd buy
   Snippet: But what's really impressive is that this portable charger can output up to 200W. And if you act quickly, you can save $40 on it over at Amazon. Price...


--- Testing Query: 'market trends in electronic vehicle charging' ---
1. [Score: 0.0164] [Africa] Wireless EV Chargers Market Size (USD 200.2 million by 2031 with a

In [ ]:
#HERE